# 한국인 + 외국인 발화 국적/언어권 분류 데모

이 노트북은 **Speech Classification 기반 멀티모달 발음 평가 및 교정 모델** 개발의 1단계 테스트다.

현재 범위:

```text
test 음성 랜덤 선택
  -> WAV 로딩 / mono / 16 kHz
  -> 처음 16초 사용
  -> frozen XLS-R 2B feature 추출 + mean pooling
  -> feature 표준화
  -> 32-class linear classifier
  -> Top-5 예측 및 추론 시간 출력
```

출력 클래스는 기존 외국인 언어권 31개와 `한국어_모국어`를 합친 32개다. 이 모델은 국적을 법적·개인적 속성으로 확정하는 모델이 아니라, **발화에 나타난 모국어/언어권 특성을 분류하는 실험 모델**로 해석해야 한다.


## 1. 실행 환경

권장 kernel: `$HOME/miniconda3/envs/fairseq_xlsr/bin/python`

XLS-R 2B checkpoint가 약 25 GB이므로 최초 모델 로딩은 1~2분 정도 걸릴 수 있다. 이후 출력되는 `inference_total_sec`은 모델 로딩 시간을 제외한 warm inference 시간이다.


In [1]:
from pathlib import Path
import sys
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import Audio, display

PROJECT_ROOT = Path('$HOME')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from country_classifier.audio import load_wav_mono, read_wav_info
from country_classifier.model import build_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAMPLE_RATE = 16_000
MAX_SEC = 16.0

XLSR_CKPT = PROJECT_ROOT / 'models/xlsr2_2B_1000k.pt'
FAIRSEQ_ROOT = PROJECT_ROOT / 'fairseq'
HEAD_CKPT = PROJECT_ROOT / 'taeho/policy32_othergpu_split_16s_probe/linear_best.pt'
TEST_CSV = PROJECT_ROOT / 'taeho/policy32_othergpu_split_seed42/test.csv'

print('device:', DEVICE)
if DEVICE == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))
print('max_sec:', MAX_SEC)
print('head:', HEAD_CKPT)


device: cuda
gpu: NVIDIA A100-SXM4-80GB
max_sec: 16.0
head: $NATIONALITY_OUTPUTS/policy32_othergpu_split_16s_probe/linear_best.pt


## 2. XLS-R와 32-class linear head 로딩

학습 때 사용한 train feature 평균/표준편차도 checkpoint에서 불러와 동일하게 표준화한다.


In [2]:
load_started = time.perf_counter()

head_payload = torch.load(HEAD_CKPT, map_location='cpu')
labels = list(head_payload['labels'])
num_classes = len(labels)

classifier = nn.Linear(1920, num_classes)
classifier.load_state_dict(head_payload['head'])
classifier = classifier.to(DEVICE).eval()

x_mean = torch.as_tensor(head_payload['mu'], dtype=torch.float32, device=DEVICE)
x_std = torch.as_tensor(head_payload['sd'], dtype=torch.float32, device=DEVICE)

xlsr_model = build_model(
    backend='fairseq_xlsr',
    num_classes=num_classes,
    checkpoint_path=XLSR_CKPT,
    fairseq_root=FAIRSEQ_ROOT,
    freeze_encoder=True,
).to(DEVICE).eval()

model_load_sec = time.perf_counter() - load_started
print(f'model loaded: {model_load_sec:.2f} sec')
print(f'classes: {num_classes}')
print('last class:', labels[-1])


INFO:fairseq.tasks.text_to_speech:Please install tensorboardX: pip install tensorboardX
$HOME/miniconda3/envs/fairseq_xlsr/lib/python3.8/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


model loaded: 41.42 sec
classes: 32
last class: 한국어_모국어


## 3. 추론 함수

시간 항목:

- `audio_load_preprocess_sec`: WAV 로딩, mono/16 kHz 변환, 16초 crop
- `xlsr_forward_pool_sec`: GPU 전송, XLS-R forward, mean pooling
- `classifier_sec`: 표준화, linear head, softmax/Top-5
- `inference_total_sec`: 위 전체 합계. 모델 로딩 및 웹 업로드 시간은 제외


In [3]:
def _sync_cuda():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()


def predict_audio(audio_path, top_k=5, max_sec=MAX_SEC):
    audio_path = Path(audio_path)
    total_start = time.perf_counter()

    load_start = time.perf_counter()
    waveform, sample_rate = load_wav_mono(audio_path, target_sample_rate=SAMPLE_RATE)
    original_samples = int(waveform.numel())
    max_samples = int(max_sec * SAMPLE_RATE)
    waveform = waveform[:max_samples].contiguous()
    used_duration_sec = waveform.numel() / SAMPLE_RATE
    audio_load_preprocess_sec = time.perf_counter() - load_start

    waveforms = waveform.unsqueeze(0).to(DEVICE, non_blocking=True)
    padding_mask = torch.zeros_like(waveforms, dtype=torch.bool, device=DEVICE)

    _sync_cuda()
    encoder_start = time.perf_counter()
    with torch.no_grad():
        if DEVICE == 'cuda':
            with torch.autocast(device_type='cuda'):
                features, feature_mask = xlsr_model._extract_features(waveforms, padding_mask)
                pooled = xlsr_model._pool(features, feature_mask)
        else:
            features, feature_mask = xlsr_model._extract_features(waveforms, padding_mask)
            pooled = xlsr_model._pool(features, feature_mask)
    _sync_cuda()
    xlsr_forward_pool_sec = time.perf_counter() - encoder_start

    classifier_start = time.perf_counter()
    with torch.no_grad():
        normalized = (pooled.float() - x_mean) / x_std
        logits = classifier(normalized)
        probabilities = torch.softmax(logits, dim=-1)[0]
        scores, indices = probabilities.topk(min(top_k, num_classes))
    _sync_cuda()
    classifier_sec = time.perf_counter() - classifier_start
    inference_total_sec = time.perf_counter() - total_start

    top_predictions = [
        {
            'rank': rank,
            'label_id': int(index),
            'label': labels[int(index)],
            'probability': float(score),
        }
        for rank, (score, index) in enumerate(zip(scores.cpu(), indices.cpu()), start=1)
    ]

    return {
        'audio_path': str(audio_path),
        'original_duration_sec': original_samples / SAMPLE_RATE,
        'used_duration_sec': used_duration_sec,
        'predicted_label': top_predictions[0]['label'],
        'predicted_label_id': top_predictions[0]['label_id'],
        'top_predictions': top_predictions,
        'timing': {
            'audio_load_preprocess_sec': audio_load_preprocess_sec,
            'xlsr_forward_pool_sec': xlsr_forward_pool_sec,
            'classifier_sec': classifier_sec,
            'inference_total_sec': inference_total_sec,
            'realtime_factor': inference_total_sec / max(used_duration_sec, 1e-9),
        },
    }


def result_tables(result):
    predictions = pd.DataFrame(result['top_predictions'])
    predictions['probability_percent'] = predictions['probability'] * 100
    predictions = predictions[['rank', 'label_id', 'label', 'probability_percent']]
    timing = pd.DataFrame([result['timing']]).T.reset_index()
    timing.columns = ['stage', 'seconds']
    return predictions, timing


## 4. Test에서 랜덤 음성 선택

`SAMPLE_GROUP`을 다음 중 하나로 바꿀 수 있다.

- `'all'`: 전체 test에서 선택
- `'foreign'`: 외국인 test에서 선택
- `'korean'`: 한국어 test에서 선택

`RANDOM_SEED = None`이면 실행할 때마다 다른 음성이 선택된다.


In [4]:
SAMPLE_GROUP = 'all'  # 'all', 'foreign', 'korean'
RANDOM_SEED = None

test_df = pd.read_csv(TEST_CSV)
if SAMPLE_GROUP == 'foreign':
    candidates = test_df[test_df['label_id'] != 31]
elif SAMPLE_GROUP == 'korean':
    candidates = test_df[test_df['label_id'] == 31]
elif SAMPLE_GROUP == 'all':
    candidates = test_df
else:
    raise ValueError("SAMPLE_GROUP must be 'all', 'foreign', or 'korean'")

random_state = RANDOM_SEED if RANDOM_SEED is not None else random.SystemRandom().randint(0, 2**32 - 1)
sample = candidates.sample(n=1, random_state=random_state).iloc[0]
audio_path = Path(sample['audio_path'])
assert audio_path.exists(), audio_path

print('random_state:', random_state)
print('true label:', sample['label_name'], f"(id={sample['label_id']})")
print('speaker:', sample['speaker_id'])
print('manifest duration:', sample['duration_sec'])
print('audio:', audio_path)
display(Audio(filename=str(audio_path)))


random_state: 3850587152
true label: 한국어_모국어 (id=31)
speaker: KOR_pingi5
manifest duration: 5.41
audio: $SPEECH_AI_DATA/aihub/001.자유대화(일반남여)/01.데이터/1.Training/2.원천데이터_0913_add/2.음성수집도구/일반남여_일반통합11_F_pingi5_35_수도권_실내/일반남여_일반통합11_F_pingi5_35_수도권_실내_19257.wav


## 5. 랜덤 음성 분류 및 추론 속도


In [5]:
result = predict_audio(audio_path, top_k=5, max_sec=MAX_SEC)
predictions, timing = result_tables(result)

true_label = str(sample['label_name'])
top5_labels = predictions['label'].tolist()

print('true label       :', true_label)
print('predicted label  :', result['predicted_label'])
print('top1 correct     :', result['predicted_label'] == true_label)
print('top5 correct     :', true_label in top5_labels)
print(f"audio duration   : {result['original_duration_sec']:.3f} sec")
print(f"used duration    : {result['used_duration_sec']:.3f} sec")
print(f"inference total  : {result['timing']['inference_total_sec']:.4f} sec")
print(f"realtime factor  : {result['timing']['realtime_factor']:.4f}")

display(predictions.style.format({'probability_percent': '{:.3f}%'}))
display(timing.style.format({'seconds': '{:.6f}'}))


$HOME/miniconda3/envs/fairseq_xlsr/lib/python3.8/site-packages/torch/nn/modules/conv.py:306: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv1d(input, weight, bias, self.stride,


true label       : 한국어_모국어
predicted label  : 한국어_모국어
top1 correct     : True
top5 correct     : True
audio duration   : 5.410 sec
used duration    : 5.410 sec
inference total  : 0.3148 sec
realtime factor  : 0.0582


,rank,label_id,label,probability_percent
0,1,31,한국어_모국어,100.000%
1,2,19,타밀어,0.000%
2,3,0,국가명_불명확,0.000%
3,4,29,희소_인도유럽_기타,0.000%
4,5,15,종카어,0.000%


,stage,seconds
0,audio_load_preprocess_sec,0.000898
1,xlsr_forward_pool_sec,0.281106
2,classifier_sec,0.028776
3,inference_total_sec,0.314809
4,realtime_factor,0.058192


## 6. 여러 랜덤 test 음성 latency benchmark

첫 1회는 CUDA kernel 초기화 영향을 줄이기 위한 warm-up으로 사용하고 집계에서 제외한다. XLS-R 2B 추론이므로 `BENCHMARK_SAMPLES`를 크게 설정하면 오래 걸릴 수 있다.


In [6]:
BENCHMARK_SAMPLES = 10
BENCHMARK_GROUP = 'all'  # 'all', 'foreign', 'korean'
BENCHMARK_SEED = 42

if BENCHMARK_GROUP == 'foreign':
    benchmark_pool = test_df[test_df['label_id'] != 31]
elif BENCHMARK_GROUP == 'korean':
    benchmark_pool = test_df[test_df['label_id'] == 31]
else:
    benchmark_pool = test_df

benchmark_rows = benchmark_pool.sample(
    n=min(BENCHMARK_SAMPLES, len(benchmark_pool)),
    random_state=BENCHMARK_SEED,
).reset_index(drop=True)

# Warm-up: 집계에서 제외
_ = predict_audio(benchmark_rows.iloc[0]['audio_path'], top_k=5, max_sec=MAX_SEC)

benchmark_results = []
for index, row in benchmark_rows.iterrows():
    output = predict_audio(row['audio_path'], top_k=5, max_sec=MAX_SEC)
    top5_labels = [item['label'] for item in output['top_predictions']]
    benchmark_results.append({
        'index': index,
        'true_label': row['label_name'],
        'predicted_label': output['predicted_label'],
        'top1_correct': output['predicted_label'] == row['label_name'],
        'top5_correct': row['label_name'] in top5_labels,
        'audio_sec': output['original_duration_sec'],
        'used_sec': output['used_duration_sec'],
        **output['timing'],
    })

benchmark_df = pd.DataFrame(benchmark_results)
display(benchmark_df)

summary = pd.DataFrame({
    'metric': [
        'samples', 'top1_accuracy', 'top5_accuracy',
        'mean_total_sec', 'median_total_sec', 'p95_total_sec',
        'mean_xlsr_sec', 'mean_classifier_sec', 'mean_realtime_factor',
    ],
    'value': [
        len(benchmark_df),
        benchmark_df['top1_correct'].mean(),
        benchmark_df['top5_correct'].mean(),
        benchmark_df['inference_total_sec'].mean(),
        benchmark_df['inference_total_sec'].median(),
        benchmark_df['inference_total_sec'].quantile(0.95),
        benchmark_df['xlsr_forward_pool_sec'].mean(),
        benchmark_df['classifier_sec'].mean(),
        benchmark_df['realtime_factor'].mean(),
    ],
})
display(summary.style.format({'value': '{:.6f}'}))


,index,true_label,predicted_label,top1_correct,top5_correct,audio_sec,used_sec,audio_load_preprocess_sec,xlsr_forward_pool_sec,classifier_sec,inference_total_sec,realtime_factor
0,0,일본어,일본어,True,True,24.119750,16.000000,0.000586,0.080740,0.000226,0.081736,0.005109
1,1,베트남어,베트남어,True,True,11.759750,11.759750,0.002558,0.082406,0.000217,0.085377,0.007260
2,2,중국어권,중국어권,True,True,13.199750,13.199750,0.001074,0.075217,0.000220,0.076669,0.005808
3,3,한국어_모국어,한국어_모국어,True,True,4.158625,4.158625,0.000253,0.067072,0.000203,0.067642,0.016265
4,4,미얀마어,미얀마어,True,True,11.839750,11.839750,0.002819,0.071903,0.000214,0.075130,0.006346
5,5,한국어_모국어,한국어_모국어,True,True,7.678125,7.678125,0.000298,0.064877,0.000202,0.065503,0.008531
6,6,한국어_모국어,한국어_모국어,True,True,3.924375,3.924375,0.000240,0.067357,0.000230,0.067937,0.017312
7,7,일본어,일본어,True,True,11.684875,11.684875,0.000409,0.071861,0.000213,0.072628,0.006216
8,8,일본어,일본어,True,True,10.281437,10.281437,0.000373,0.070176,0.000209,0.070896,0.006895
9,9,스페인어,태국어,False,True,13.739750,13.739750,0.003325,0.071261,0.000234,0.075039,0.005461


,metric,value
0,samples,10.000000
1,top1_accuracy,0.900000
2,top5_accuracy,1.000000
3,mean_total_sec,0.073856
4,median_total_sec,0.073833
5,p95_total_sec,0.083738
6,mean_xlsr_sec,0.072287
7,mean_classifier_sec,0.000217
8,mean_realtime_factor,0.008520


## 7. 임의 WAV 파일 테스트

웹 업로드 또는 별도 녹음 파일을 시험할 때 아래 경로만 바꾼다. 학습 조건과 맞추기 위해 처음 16초를 사용한다.


In [7]:
# CUSTOM_AUDIO_PATH = Path('/path/to/custom.wav')
# custom_result = predict_audio(CUSTOM_AUDIO_PATH, top_k=5, max_sec=MAX_SEC)
# custom_predictions, custom_timing = result_tables(custom_result)
# display(Audio(filename=str(CUSTOM_AUDIO_PATH)))
# display(custom_predictions.style.format({'probability_percent': '{:.3f}%'}))
# display(custom_timing.style.format({'seconds': '{:.6f}'}))


## 해석 시 주의사항

- 한국어와 외국인 데이터는 서로 다른 AIHub 수집 파이프라인에서 왔다. 모델이 발음뿐 아니라 녹음 장비/환경 차이를 활용할 수 있다.
- 현재 test는 speaker-disjoint이지만, 실제 웹 환경 성능을 보장하지 않는다.
- 국적·출신을 확정하는 용도로 사용하지 않는다. 발음 평가 모델의 보조적인 언어권 추정 결과로 사용한다.
- 다음 단계에서는 독립 웹 녹음 데이터로 검증하고, 음소/운율/억양 feature와 결합한 멀티모달 발음 평가·교정 모델로 확장한다.
